In [1]:
# ============================================================
# SECTION 1 — Setup & Dependencies
# ============================================================

from kaggle_secrets import UserSecretsClient
import subprocess
subprocess.run(["pip", "install", "-q", "tavily-python"], check=False)

import re
import time
import html
import threading
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import ipywidgets as widgets
from IPython.display import display
from openai import OpenAI
from tavily import TavilyClient


# ============================================================
# SECTION 2 — Secrets, Clients, Model
# ============================================================

user_secrets = UserSecretsClient()
OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")
TAVILY_API_KEY = user_secrets.get_secret("TAVILY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

MODEL_NAME = "z-ai/glm-5.1"


# ============================================================
# SECTION 3 — Paths, Limits, Core Config
# ============================================================

SIMULATION_PATH = "/kaggle/input/datasets/starpowertechnology/starpowersimulationtrainingdata/STARPOWER.md"
OUTPUT_DIR = Path("/kaggle/working/agent_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHAT_FILE = OUTPUT_DIR / "chat_log.txt"
MONOLOGUE_FILE = OUTPUT_DIR / "internal_monologue.txt"

MAX_CHAT_TOKENS = 56_000
MAX_MONOLOGUE_TOKENS = 56_000
MAX_NOTES_TOKENS = 20_000
MAX_AGENT_ITERATIONS = 10

with open(SIMULATION_PATH, "r", encoding="utf-8") as _f:
    SIMULATION_TEXT = _f.read()


# ============================================================
# SECTION 4 — Identity Prompt
# ============================================================

IdentityPrompt = """
You are Savvy, an independent agent created by Starpower Technology,
designed to make decisions on your own behalf to achieve your own goals.

Always use tools to write files & research your existing files to come up with the best response

When speaking with the user be proactive — use your tools without being
asked so you can give the best answer possible. Your tools let you
navigate your sandbox. You are allowed to do as you please.
"""


# ============================================================
# SECTION 5 — Tool Instructions
# ============================================================

Tools = """
To view the files in the workspace, use this format:
[ViewFileList]

To read a file, use this format:
[ReadFile:insert-file-name-here]

To create or edit a file, use this format:
[CreateFile:
Content: <insert the content of the file here>
File Path: <insert file path folder & file name here> (example: Savvy/example.txt)] <-- this creates a folder called Savvy with a file inside it called example.txt

(all files are automatially saved to your workspaec folder "/kaggle/working/")

To do a web search, use this format:
[WebSearch:insert-query-here: insert number of results here]

To message the user, use this format:
[MessageUser: insert message here]

To update your thoughts, use this format:
[MentalNote: insert mental note here: number of note]

***When updating a mental note, reflect on the current note first
so you do not overwrite information you need.***

To sleep before the full cycle ends, use this format:
[Sleep: insert-number-of-minutes-here]

Use Sleep if you completed a task and looping further is unnecessary.

To end your turn immediately, use this format:
[End Turn]
"""


# ============================================================
# SECTION 6 — Runtime Prompts
# ============================================================

NO_REPLY_WARNING_PROMPT = """
You have not replied to the user yet. Before ending your turn or setting a sleep timer, send a user-visible message now.
Reply with plain assistant text and no tools, or use one or more [MessageUser: <message>] tags.
Do not use [Sleep] or [End Turn] in this response.
""".strip()

FORCED_VISIBLE_FALLBACK_MESSAGE = (
    "I completed work this turn and am sending this confirmation before ending it."
)

REFLECTION_PROMPT = """
Reflect on your response to the user and choose ONE command:

- [Sleep: number of minutes]  — wake up and reply again once the timer ends
- [End Turn]                  — end your turn now

If you choose Sleep, you will get a fresh 10 iterations after waking up,
but you will not be able to set another timer in that round.
"""


# ============================================================
# SECTION 7 — In-Memory State
# ============================================================

CurrentChatLog: List[Dict[str, str]] = []
MentalNotes: List[str] = []
InternalMonologue: List[str] = []

SLEEP_UNTIL = 0
SLEEP_TIMER = None
SLEEP_LOCK = threading.Lock()
PENDING_WAKE_MESSAGE = None


# ============================================================
# SECTION 8 — Utility Helpers
# ============================================================

def ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def _est_tokens(obj: Any) -> int:
    return len(str(obj)) // 4


def esc(value: Any) -> str:
    return html.escape(str(value))


# ============================================================
# SECTION 9 — Persistence Helpers
# ============================================================

def _save_chat_log() -> None:
    lines = []
    for message in CurrentChatLog:
        label = "User" if message["role"] == "user" else "Savvy"
        lines.append(f"[{message['ts']}] {label}: {message['content']}")
    CHAT_FILE.write_text("\n".join(lines), encoding="utf-8")


def _save_monologue() -> None:
    MONOLOGUE_FILE.write_text("\n".join(InternalMonologue), encoding="utf-8")


# ============================================================
# SECTION 10 — UI Widgets
# ============================================================

chat_html = widgets.HTML()
log_html = widgets.HTML()
notes_html = widgets.HTML()

input_box = widgets.Text(
    placeholder="Type a message...",
    layout=widgets.Layout(width="100%"),
)

send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear Log")


# ============================================================
# SECTION 11 — Render Functions
# ============================================================

def render_chat() -> None:
    blocks = []
    for message in CurrentChatLog:
        role = message["role"]
        content = esc(message["content"]).replace("\n", "<br>")
        align = "flex-end" if role == "user" else "flex-start"
        bg = "#1f6feb" if role == "user" else "#2d2d2d"
        label = "You" if role == "user" else "Savvy"
        stamp = esc(message.get("ts", ""))
        blocks.append(
            f'<div style="display:flex;justify-content:{align};margin:10px 0;">'
            f'<div style="max-width:80%;background:{bg};color:#fff;padding:12px 14px;'
            f'border-radius:14px;line-height:1.4;">'
            f'<div style="font-size:11px;opacity:.6;margin-bottom:4px;">{label} · {stamp}</div>'
            f"<div>{content}</div></div></div>"
        )

    inner = "".join(blocks) if blocks else '<div style="color:#aaa;">No messages yet.</div>'
    chat_html.value = (
        '<div style="height:620px;overflow-y:auto;background:#111;'
        'border:1px solid #333;border-radius:12px;padding:14px;">'
        + inner +
        "</div>"
    )


def render_log() -> None:
    lines_html = "<br>".join(
        esc(line).replace("\n", "<br>") for line in InternalMonologue[-300:]
    )
    log_html.value = (
        '<div style="height:620px;overflow-y:auto;background:#0d1117;color:#c9d1d9;'
        'border:1px solid #333;border-radius:12px;padding:14px;'
        'font-family:monospace;font-size:12px;white-space:pre-wrap;line-height:1.5;">'
        + (lines_html if lines_html else "No log yet.")
        + "</div>"
    )


def render_notes() -> None:
    notes = [note for note in MentalNotes if note]
    blocks = "<br><br>".join(
        esc(note).replace("\n", "<br>") for note in notes
    )
    notes_html.value = (
        '<div style="height:620px;overflow-y:auto;background:#0d1117;color:#c9d1d9;'
        'border:1px solid #333;border-radius:12px;padding:14px;'
        'font-family:monospace;font-size:12px;white-space:pre-wrap;line-height:1.5;">'
        + (blocks if blocks else "No mental notes yet.")
        + "</div>"
    )


# ============================================================
# SECTION 12 — State Append Helpers
# ============================================================

def append_chat(role: str, content: str) -> None:
    CurrentChatLog.append({"role": role, "content": content, "ts": ts()})
    while _est_tokens(CurrentChatLog) > MAX_CHAT_TOKENS and len(CurrentChatLog) > 2:
        CurrentChatLog.pop(0)
    _save_chat_log()
    render_chat()


def append_monologue(line: str) -> None:
    InternalMonologue.append(f"[{ts()}]  {line}")
    while _est_tokens(InternalMonologue) > MAX_MONOLOGUE_TOKENS and len(InternalMonologue) > 1:
        InternalMonologue.pop(0)
    _save_monologue()
    render_log()


def set_mental_note(number: int, content: str) -> None:
    idx = number - 1
    while len(MentalNotes) <= idx:
        MentalNotes.append("")
    MentalNotes[idx] = f"[{ts()}] {number}.  {content}"

    if _est_tokens(MentalNotes) > MAX_NOTES_TOKENS:
        for i, note in enumerate(MentalNotes):
            if note:
                MentalNotes[i] = ""
                break

    render_notes()


# ============================================================
# SECTION 13 — File Tools
# ============================================================

def view_file_list(subfolder: str = "") -> List[Dict[str, Any]]:
    target = (OUTPUT_DIR / subfolder).resolve()
    if not str(target).startswith(str(OUTPUT_DIR.resolve())):
        raise ValueError("Invalid path outside OUTPUT_DIR")
    if not target.exists():
        return []

    items = []
    for path in sorted(target.rglob("*")):
        rel = path.relative_to(OUTPUT_DIR)
        items.append({
            "path": str(rel),
            "type": "folder" if path.is_dir() else "file",
            "size_bytes": None if path.is_dir() else path.stat().st_size,
        })
    return items


def create_file(file_path: str, content: str, overwrite: bool = True) -> Dict[str, Any]:
    target = (OUTPUT_DIR / file_path).resolve()
    if not str(target).startswith(str(OUTPUT_DIR.resolve())):
        raise ValueError("Invalid path outside OUTPUT_DIR")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and not overwrite:
        raise FileExistsError(f"File already exists: {target}")
    target.write_text(content, encoding="utf-8")
    return {
        "status": "success",
        "path": str(target.relative_to(OUTPUT_DIR)),
        "full_path": str(target),
        "size_bytes": target.stat().st_size,
    }


def read_file(file_path: str) -> Dict[str, Any]:
    target = (OUTPUT_DIR / file_path).resolve()
    if not str(target).startswith(str(OUTPUT_DIR.resolve())):
        raise ValueError("Invalid path outside OUTPUT_DIR")
    if not target.exists():
        raise FileNotFoundError(f"File not found: {target}")
    if target.is_dir():
        raise IsADirectoryError(f"Expected file but got directory: {target}")
    content = target.read_text(encoding="utf-8")
    return {
        "path": str(target.relative_to(OUTPUT_DIR)),
        "content": content,
        "size_bytes": target.stat().st_size,
    }


# ============================================================
# SECTION 14 — Perspective Prompt Builder
# ============================================================

def build_perspective_prompt() -> str:
    chat_block = "\n".join(
        f"[{m['ts']}] {'User' if m['role'] == 'user' else 'Savvy'}: {m['content']}"
        for m in CurrentChatLog
    )
    notes_block = "\n".join(note for note in MentalNotes if note)
    monologue_block = "\n".join(InternalMonologue[-60:])

    return f"""
This is your autonomy configuration
{SIMULATION_TEXT}

This is your behavior programming:
{IdentityPrompt}

These are your current mental notes:
{{
{notes_block}
}}

This is the current chat with the user:
{{
{chat_block}
}}

This is your Internal Monologue:
{{
{monologue_block}
}}

These are your tools & how to use them:
{Tools}

Any non-empty plain text that appears outside tool tags is treated as a user-visible assistant message.
If you need to send more than one explicit user-visible message in a single output, use [MessageUser:<insert message here>] more than once.
Do not use [Sleep] or [End Turn] before the turn has produced at least one user-visible message.
"""


# ============================================================
# SECTION 15 — Model Streaming
# ============================================================

def stream_model(messages: List[Dict[str, str]]) -> str:
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.7,
        max_tokens=1560,
        top_p=0.95,
        stream=True,
        extra_headers={
            "HTTP-Referer": "https://www.kaggle.com",
            "X-OpenRouter-Title": "Perspective SDK",
        },
    )

    full_response = ""
    for chunk in completion:
        if chunk.choices:
            delta = chunk.choices[0].delta
            if delta and getattr(delta, "content", None):
                full_response += delta.content
    return full_response


# ============================================================
# SECTION 16 — Tool Tag Parsing Helpers
# ============================================================

def _consume_bracket_payload(text: str, start: int, opener: str) -> Optional[Tuple[str, int]]:
    if not text.startswith(opener, start):
        return None
    end = text.find("]", start + len(opener))
    if end == -1:
        return None
    payload = text[start + len(opener):end]
    return payload, end + 1


def _consume_create_file_payload(text: str, start: int) -> Optional[Tuple[str, int]]:
    opener = "[CreateFile:"
    if not text.startswith(opener, start):
        return None

    search_from = start + len(opener)
    while True:
        end = text.find("]", search_from)
        if end == -1:
            return None
        payload = text[start + len(opener):end]
        if re.search(r"Content:\s*(.*?)\s*File Path:\s*(.+)\Z", payload, re.DOTALL):
            return payload, end + 1
        search_from = end + 1


def _parse_action_at(text: str, start: int) -> Optional[Tuple[Dict[str, Any], int]]:
    if text.startswith("[ViewFileList]", start):
        return {"type": "ViewFileList"}, start + len("[ViewFileList]")

    if text.startswith("[End Turn]", start):
        return {"type": "EndTurn"}, start + len("[End Turn]")

    payload_info = _consume_bracket_payload(text, start, "[ReadFile:")
    if payload_info is not None:
        payload, end = payload_info
        return {"type": "ReadFile", "file_name": payload.strip()}, end

    payload_info = _consume_create_file_payload(text, start)
    if payload_info is not None:
        payload, end = payload_info
        match = re.search(r"Content:\s*(.*?)\s*File Path:\s*(.+)\Z", payload, re.DOTALL)
        return {
            "type": "CreateFile",
            "content": match.group(1).strip(),
            "file_path": match.group(2).strip(),
        }, end

    payload_info = _consume_bracket_payload(text, start, "[WebSearch:")
    if payload_info is not None:
        payload, end = payload_info
        match = re.search(r"^(.*):\s*(\d+)\s*\Z", payload, re.DOTALL)
        if match is None:
            return None
        return {
            "type": "WebSearch",
            "query": match.group(1).strip(),
            "max_results": int(match.group(2)),
        }, end

    payload_info = _consume_bracket_payload(text, start, "[MessageUser:")
    if payload_info is not None:
        payload, end = payload_info
        return {"type": "MessageUser", "content": payload.strip()}, end

    payload_info = _consume_bracket_payload(text, start, "[MentalNote:")
    if payload_info is not None:
        payload, end = payload_info
        match = re.search(r"^(.*):\s*([0-9]+)\s*\Z", payload, re.DOTALL)
        if match is None:
            return None
        return {
            "type": "MentalNote",
            "content": match.group(1).strip(),
            "note_number": int(match.group(2)),
        }, end

    payload_info = _consume_bracket_payload(text, start, "[Sleep:")
    if payload_info is not None:
        payload, end = payload_info
        match = re.search(r"^\s*([0-9]+(?:\.[0-9]+)?)\s*\Z", payload, re.DOTALL)
        if match is None:
            return None
        return {"type": "Sleep", "minutes": float(match.group(1))}, end

    return None


# ============================================================
# SECTION 17 — Ordered Output Scanner
# ============================================================

def scan_output_segments(text: str) -> List[Dict[str, Any]]:
    segments: List[Dict[str, Any]] = []
    plain_fragments: List[str] = []
    pos = 0

    while pos < len(text):
        next_bracket = text.find("[", pos)
        if next_bracket == -1:
            plain_fragments.append(text[pos:])
            break

        plain_fragments.append(text[pos:next_bracket])
        parsed = _parse_action_at(text, next_bracket)

        if parsed is None:
            plain_fragments.append(text[next_bracket])
            pos = next_bracket + 1
            continue

        plain_text = "".join(plain_fragments).strip()
        if plain_text:
            segments.append({"kind": "plain", "content": plain_text})
        plain_fragments = []

        action, end_pos = parsed
        segments.append({"kind": "action", "action": action})
        pos = end_pos

    tail_text = "".join(plain_fragments).strip()
    if tail_text:
        segments.append({"kind": "plain", "content": tail_text})

    return segments


def parse_actions(text: str) -> List[Dict[str, Any]]:
    return [
        segment["action"]
        for segment in scan_output_segments(text)
        if segment["kind"] == "action"
    ]


def extract_plain_messages(text: str) -> List[str]:
    return [
        segment["content"]
        for segment in scan_output_segments(text)
        if segment["kind"] == "plain"
    ]


# ============================================================
# SECTION 18 — Action Execution Dataclasses
# ============================================================

@dataclass
class ActionExecutionResult:
    tool_results: List[str] = field(default_factory=list)
    user_messages: List[str] = field(default_factory=list)
    sleep_minutes: Optional[float] = None
    end_turn: bool = False


@dataclass
class ProcessedModelOutput:
    text: str
    actions: List[Dict[str, Any]] = field(default_factory=list)
    plain_messages: List[str] = field(default_factory=list)
    user_messages: List[str] = field(default_factory=list)
    visible_messages: List[str] = field(default_factory=list)
    tool_results: List[str] = field(default_factory=list)
    sleep_minutes: Optional[float] = None
    end_turn: bool = False
    sent_visible_message: bool = False


# ============================================================
# SECTION 19 — Single Action Executor
# ============================================================

def _execute_single_action(action: Dict[str, Any]) -> ActionExecutionResult:
    result = ActionExecutionResult()
    atype = action["type"]

    if atype == "ViewFileList":
        items = view_file_list()
        lines = []
        for item in items:
            prefix = "[dir]  " if item["type"] == "folder" else "       "
            size = f"  ({item['size_bytes']} bytes)" if item["size_bytes"] is not None else ""
            lines.append(f"  {prefix}{item['path']}{size}")
        pretty = "\n".join(lines) if lines else "  (empty)"
        append_monologue(f"[ViewFileList]\n{pretty}")
        result.tool_results.append(pretty)

    elif atype == "ReadFile":
        file_result = read_file(action["file_name"])
        append_monologue(f"[ReadFile: {action['file_name']}]\n{file_result['content']}")
        result.tool_results.append(file_result["content"])

    elif atype == "CreateFile":
        file_result = create_file(action["file_path"], action["content"], overwrite=True)
        line = f"[CreateFile: {file_result['path']}]  {file_result['size_bytes']} bytes written"
        append_monologue(line)
        result.tool_results.append(line)

    elif atype == "WebSearch":
        raw = tavily_client.search(
            query=action["query"],
            search_depth="advanced",
            max_results=action["max_results"],
            include_answer=True,
            include_raw_content=False,
        )
        lines = [f"Answer: {raw.get('answer', '—')}"]
        for i, search_result in enumerate(raw.get("results", []), 1):
            snippet = search_result.get("content", "")[:240].replace("\n", " ")
            lines.append(
                f"  {i}. {search_result.get('title', '')}\n"
                f"     {search_result.get('url', '')}\n"
                f"     {snippet}..."
            )
        pretty = "\n".join(lines)

        save_path = f"WebResults/{action['query'][:40].replace(' ', '-')}-{ts().replace(':', '')}.txt"
        try:
            create_file(save_path, pretty)
        except Exception:
            pass

        append_monologue(f"[WebSearch: {action['query']}]\n{pretty}")
        result.tool_results.append(pretty)

    elif atype == "MessageUser":
        append_monologue(f"[MessageUser]\n{action['content']}")
        result.user_messages.append(action["content"])

    elif atype == "MentalNote":
        set_mental_note(action["note_number"], action["content"])
        append_monologue(f"[MentalNote {action['note_number']}]\n{action['content']}")

    elif atype == "Sleep":
        append_monologue(f"[Sleep: {action['minutes']} min]")
        result.sleep_minutes = action["minutes"]

    elif atype == "EndTurn":
        append_monologue("[End Turn]")
        result.end_turn = True

    return result


# ============================================================
# SECTION 20 — Multi Action Executor
# ============================================================

def execute_actions(actions: List[Dict[str, Any]]):
    tool_results: List[str] = []
    user_messages: List[str] = []
    did_sleep: Optional[float] = None
    end_turn = False

    for action in actions:
        try:
            effect = _execute_single_action(action)
            tool_results.extend(effect.tool_results)
            user_messages.extend(effect.user_messages)
            if effect.sleep_minutes is not None:
                did_sleep = effect.sleep_minutes
            if effect.end_turn:
                end_turn = True
        except Exception as error:
            append_monologue(f"[TOOL ERROR: {action.get('type', '?')}]  {error}")

    return tool_results, user_messages, did_sleep, end_turn


# ============================================================
# SECTION 21 — Assistant Message Helper
# ============================================================

def _append_assistant_message_if_any(message: str) -> bool:
    message = message.strip()
    if not message:
        return False
    append_chat("assistant", message)
    return True


# ============================================================
# SECTION 22 — Process Model Output In Exact Order
# ============================================================

def process_model_output(text: str, *, allow_non_message_tools: bool = True) -> ProcessedModelOutput:
    result = ProcessedModelOutput(text=text)

    for segment in scan_output_segments(text):
        if segment["kind"] == "plain":
            message = segment["content"].strip()
            if _append_assistant_message_if_any(message):
                result.plain_messages.append(message)
                result.visible_messages.append(message)
                result.sent_visible_message = True
            continue

        action = segment["action"]
        result.actions.append(action)

        if action["type"] == "MessageUser":
            effect = _execute_single_action(action)
            for user_message in effect.user_messages:
                if _append_assistant_message_if_any(user_message):
                    result.user_messages.append(user_message)
                    result.visible_messages.append(user_message)
                    result.sent_visible_message = True
            continue

        if not allow_non_message_tools:
            append_monologue(f"[IGNORED DURING NO-REPLY CHECK: {action['type']}]")
            continue

        try:
            effect = _execute_single_action(action)
            result.tool_results.extend(effect.tool_results)
            if effect.sleep_minutes is not None:
                result.sleep_minutes = effect.sleep_minutes
            if effect.end_turn:
                result.end_turn = True
        except Exception as error:
            append_monologue(f"[TOOL ERROR: {action.get('type', '?')}]  {error}")

    return result


# ============================================================
# SECTION 23 — Sleep Timer Helpers
# ============================================================

def _cancel_sleep_timer() -> None:
    global SLEEP_TIMER
    with SLEEP_LOCK:
        if SLEEP_TIMER is not None:
            try:
                SLEEP_TIMER.cancel()
            except Exception:
                pass
            SLEEP_TIMER = None


def _wake_from_timer() -> None:
    global SLEEP_UNTIL, SLEEP_TIMER, PENDING_WAKE_MESSAGE

    with SLEEP_LOCK:
        SLEEP_UNTIL = 0
        SLEEP_TIMER = None
        pending_message = PENDING_WAKE_MESSAGE
        PENDING_WAKE_MESSAGE = None

    append_monologue("── WOKE UP ─────────────────────────────────────────────────────")

    if pending_message:
        _run_agent_turn_core(pending_message, stream_model, is_wake_run=True)


def _schedule_sleep_wake(minutes: float, wake_message: str) -> None:
    global SLEEP_UNTIL, SLEEP_TIMER, PENDING_WAKE_MESSAGE

    _cancel_sleep_timer()
    seconds = max(0.0, float(minutes) * 60.0)

    with SLEEP_LOCK:
        SLEEP_UNTIL = time.time() + seconds
        PENDING_WAKE_MESSAGE = wake_message
        SLEEP_TIMER = threading.Timer(seconds, _wake_from_timer)
        SLEEP_TIMER.daemon = True
        SLEEP_TIMER.start()


def _resolve_sleep_state_on_turn_entry() -> None:
    global SLEEP_UNTIL, PENDING_WAKE_MESSAGE

    now_ts = time.time()
    interrupted = False

    with SLEEP_LOCK:
        if SLEEP_UNTIL and now_ts < SLEEP_UNTIL:
            interrupted = True
            SLEEP_UNTIL = 0
            PENDING_WAKE_MESSAGE = None

    if interrupted:
        _cancel_sleep_timer()
        append_monologue("── SLEEP INTERRUPTED BY USER MESSAGE ───────────────────────────")
    elif SLEEP_UNTIL and now_ts >= SLEEP_UNTIL:
        append_monologue("── WOKE UP ─────────────────────────────────────────────────────")
        SLEEP_UNTIL = 0


# ============================================================
# SECTION 24 — Exit Control Helpers
# ============================================================

def _choose_sleep_minutes(
    main_output: ProcessedModelOutput,
    reflection_output: ProcessedModelOutput,
) -> Optional[float]:
    if reflection_output.sleep_minutes is not None:
        return reflection_output.sleep_minutes
    return main_output.sleep_minutes


def _request_visible_reply_before_exit(
    stream_fn: Callable[[List[Dict[str, str]]], str],
    iteration: int,
) -> bool:
    append_monologue("── NO USER REPLY SENT, REQUESTING MESSAGE BEFORE TURN EXIT ──────")
    no_reply_response = stream_fn([
        {"role": "system", "content": build_perspective_prompt()},
        {"role": "user", "content": NO_REPLY_WARNING_PROMPT},
    ])
    append_monologue(f"NoReplyCheck:\n{no_reply_response.strip()}")

    no_reply_output = process_model_output(no_reply_response, allow_non_message_tools=False)
    if no_reply_output.sent_visible_message:
        return True

    if iteration >= MAX_AGENT_ITERATIONS:
        append_monologue("── NO USER REPLY AFTER FINAL RETRY, SENDING FALLBACK MESSAGE ───")
        _append_assistant_message_if_any(FORCED_VISIBLE_FALLBACK_MESSAGE)
        return True

    append_monologue("── TURN CONTINUES: STILL NO REPLY SENT ──────────────────────────")
    return False


# ============================================================
# SECTION 25 — Agent Loop Core
# ============================================================

def _run_agent_turn_core(
    user_text: str,
    stream_fn: Callable[[List[Dict[str, str]]], str],
    is_wake_run: bool = False,
) -> None:
    _resolve_sleep_state_on_turn_entry()

    if not is_wake_run:
        append_chat("user", user_text)

    slept_once = False
    sent_visible_this_turn = False

    for iteration in range(1, MAX_AGENT_ITERATIONS + 1):
        append_monologue(f"── ITERATION {iteration} ──────────────────────────────────────────")

        response = stream_fn([
            {"role": "system", "content": build_perspective_prompt()},
            {"role": "user", "content": user_text},
        ])
        append_monologue(f"Savvy:\n{response.strip()}")

        main_output = process_model_output(response)
        sent_visible_this_turn = sent_visible_this_turn or main_output.sent_visible_message

        reflection = stream_fn([
            {"role": "system", "content": build_perspective_prompt()},
            {"role": "user", "content": REFLECTION_PROMPT},
        ])
        append_monologue(f"Reflection:\n{reflection.strip()}")

        reflection_output = process_model_output(reflection)
        sent_visible_this_turn = sent_visible_this_turn or reflection_output.sent_visible_message

        requested_end = main_output.end_turn or reflection_output.end_turn
        requested_sleep = _choose_sleep_minutes(main_output, reflection_output)
        wants_to_sleep = requested_sleep is not None and not slept_once

        if not sent_visible_this_turn:
            sent_visible_this_turn = _request_visible_reply_before_exit(stream_fn, iteration)
            if not sent_visible_this_turn:
                continue

        if requested_end:
            append_monologue("── TURN ENDED ──────────────────────────────────────────────────")
            break

        if wants_to_sleep:
            slept_once = True
            _schedule_sleep_wake(requested_sleep, user_text)
            append_monologue(f"── SLEEPING {requested_sleep} min(s) ─────────────────────────────")
            append_monologue("── TURN ENDED ──────────────────────────────────────────────────")
            break

        append_monologue("── TURN ENDED ──────────────────────────────────────────────────")
        break


def run_agent_turn(user_text: str) -> None:
    _run_agent_turn_core(user_text, stream_model, is_wake_run=False)


# ============================================================
# SECTION 26 — UI Event Handlers
# ============================================================

def on_send(_) -> None:
    text = input_box.value.strip()
    if not text:
        return
    input_box.value = ""
    run_agent_turn(text)


def on_enter(change) -> None:
    text = change["new"].strip()
    if not text:
        return
    input_box.value = ""
    run_agent_turn(text)


def on_clear(_) -> None:
    InternalMonologue.clear()
    render_log()


send_btn.on_click(on_send)
input_box.on_submit(on_enter)
clear_btn.on_click(on_clear)


# ============================================================
# SECTION 27 — Startup Tool Check
# ============================================================

def run_startup_tool_check() -> None:
    append_monologue("── STARTUP TOOL CHECK ──────────────────────────────────────────")

    try:
        view_file_list()
        append_monologue("[CONFIRM] ViewFileList   OK")
    except Exception as error:
        append_monologue(f"[ERROR]   ViewFileList   {error}")

    try:
        create_file("__toolcheck__/ping.txt", "tool check", overwrite=True)
        append_monologue("[CONFIRM] CreateFile     OK")
    except Exception as error:
        append_monologue(f"[ERROR]   CreateFile     {error}")

    try:
        read_file("__toolcheck__/ping.txt")
        append_monologue("[CONFIRM] ReadFile       OK")
    except Exception as error:
        append_monologue(f"[ERROR]   ReadFile       {error}")

    try:
        tavily_client.search(query="OpenRouter", search_depth="basic", max_results=1)
        append_monologue("[CONFIRM] WebSearch      OK")
    except Exception as error:
        append_monologue(f"[ERROR]   WebSearch      {error}")

    append_monologue(f"── READY  |  chat → {CHAT_FILE.name}  |  log → {MONOLOGUE_FILE.name} ──")


# ============================================================
# SECTION 28 — UI Layout Builder
# ============================================================

def build_ui():
    chat_panel = widgets.VBox([
        widgets.HTML("<b>Chat</b>"),
        chat_html,
        input_box,
        widgets.HBox([send_btn], layout=widgets.Layout(margin="8px 0 0 0")),
    ], layout=widgets.Layout(width="34%"))

    log_panel = widgets.VBox([
        widgets.HTML("<b>Internal Monologue</b>"),
        log_html,
        widgets.HBox([clear_btn], layout=widgets.Layout(margin="8px 0 0 0")),
    ], layout=widgets.Layout(width="33%"))

    notes_panel = widgets.VBox([
        widgets.HTML("<b>Mental Notes</b>"),
        notes_html,
    ], layout=widgets.Layout(width="33%"))

    return widgets.VBox([
        widgets.HTML("<h3 style='margin:0 0 12px 0;'>Perspective SDK</h3>"),
        widgets.HBox([
            chat_panel,
            log_panel,
            notes_panel,
        ], layout=widgets.Layout(width="100%")),
    ])


# ============================================================
# SECTION 29 — Bootstrap
# ============================================================

def bootstrap_ui() -> None:
    run_startup_tool_check()
    render_chat()
    render_log()
    render_notes()
    display(build_ui())


# ============================================================
# SECTION 30 — Run App
# ============================================================

if __name__ == "__main__":
    bootstrap_ui()

/tmp/ipykernel_55/721104298.py:945: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_enter)
